<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_02_optimiser_comparison_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 02 — Four Optimisers, One Problem

**Paired with L6.1 · Loss Functions and Gradients**

Notebook 01 asked where the loss comes from. This one asks how you get to the
bottom of it, and it is the notebook that earns its keep in Part 2: every PINN
in L7 to L12 is trained by **Adam first, then L-BFGS**, and this is where you
find out why that handoff exists rather than being told.

## What you will do

1. Fit the damped structural response with the same network four times,
   changing only the optimiser.
2. Sweep the learning rate and watch SGD fail in two different directions.
3. Train plain SGD on random batches of four sizes, and see what the batch
   size trades.
4. Meet L-BFGS, whose interface is different because it evaluates the loss
   more than once per step.
5. Chain Adam into L-BFGS and measure what the handoff buys.

## One thing to notice before you start

The target is **noiseless** — `core.response_dataset` adds nothing. That is
deliberate, and notebook 00 said so. With noise, every optimiser stops at the
same floor and the comparison measures the noise instead of the optimiser.
Here the floor is zero, so the differences between these four are visible for
as long as you care to train.

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
print("torch", torch.__version__, "| output dir:", core.OUTPUT_DIR)

## 1 · The problem

Two hundred noiseless samples of `exp(-0.9x) sin(4x)`, and a network with two
hidden layers of sixteen neurons. Small enough to train in seconds, wiggly enough
that a badly tuned optimiser visibly fails.

In [ ]:
# The curve to fit: a damped oscillation, 200 noiseless samples.
#   core.response_dataset(n) -> x, y
#   core.to_tensor(a) makes the (N, 1) float32 column PyTorch wants
x, y = core.response_dataset(n=200)
X, Y = core.to_tensor(x), core.to_tensor(y)
loss_fn = nn.MSELoss()

core.set_seed(0)
print("parameters in the model:", core.count_parameters(core.MLP(hidden=(16, 16))))

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(x, y, lw=1.9, color="#1f77b4")
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title("The target: a damped structural response, sampled without noise")
ax.grid(alpha=0.25)
plt.show()

### Your turn

One function, used everywhere below. Keeping the seed reset **inside** it is
what makes the four runs comparable: every optimiser starts from an identical
set of weights, so any difference you see is the optimiser and not the draw.

In [ ]:
# one training run, any optimiser -------------------------------------------
def run(make_opt, epochs=2000):
    core.set_seed(0)                     # identical initial weights, always
    model = core.MLP(hidden=(16, 16))
    opt = make_opt(model.parameters())
    history = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        opt.step()
        history.append(loss.item())
    return np.array(history), model
# ------------------------------------------------------------------------------

## 2 · Three optimisers at the same budget

Same initial weights, same two thousand epochs, same data. The only thing that
changes is the update rule.

In [ ]:
# The first-order methods, each as a factory rather than an optimiser:
# make_opt(params) -> optimiser, so every method starts from freshly
# initialised weights and the comparison is fair.
FIRST_ORDER = {
    "SGD":            lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD + momentum": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam":           lambda p: torch.optim.Adam(p, lr=0.01),
}

histories, models = {}, {}
for name, make in FIRST_ORDER.items():
    histories[name], models[name] = run(make, epochs=2000)
    print(f"  {name:<16s} final loss {histories[name][-1]:.6f}")

core.plot_curves(histories, title="Same network, same budget, three update rules")
plt.show()

**What you should see.** Three curves separated by orders of magnitude, plain
SGD highest. Momentum and Adam both get further, and Adam gets there sooner.

The gap is not that SGD is a bad algorithm. It is that plain SGD applies the
same step size to every parameter, and this loss surface is far steeper in some
directions than others — so the step that is right for the steep directions is
far too small for the flat ones.

## 3 · The learning rate decides more than the optimiser does

Before concluding anything about update rules, check whether you have merely
compared three learning rates. Sweep it.

### Your turn

In [ ]:
# the learning-rate sweep ---------------------------------------------------
LRS = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3]

def final_loss(make_opt):
    history, _ = run(make_opt, epochs=800)
    v = float(history[-1])
    return v if np.isfinite(v) else np.nan       # a diverged run IS a result

sweep = {"SGD": [], "Adam": []}
for lr in LRS:
    sweep["SGD"].append(final_loss( lambda p, lr=lr: torch.optim.SGD(p, lr=lr)))
    sweep["Adam"].append(final_loss( lambda p, lr=lr: torch.optim.Adam(p, lr=lr)))
    print(f"lr {lr:g}: SGD {sweep['SGD'][-1]}, Adam {sweep['Adam'][-1]}")
# ------------------------------------------------------------------------------

In [ ]:
# The learning-rate sweep. The lesson is the shape of each curve:
# SGD's useful range is narrow, Adam's is wide, and that width is
# most of why Adam is the default.
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for i, (name, losses) in enumerate(sweep.items()):
    ax.plot(LRS, losses, "o-", lw=1.9, ms=6, color=["#d94f2b", "#1f77b4"][i],
            label=name)
    for lr, v in zip(LRS, losses):
        if not np.isfinite(v):
            ax.plot([lr], [ax.get_ylim()[1]], "x", ms=11, mew=2.4,
                    color=["#d94f2b", "#1f77b4"][i])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("learning rate"); ax.set_ylabel("loss after 800 epochs")
ax.set_title("A cross marks a run that diverged")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

print(core.error_table(
    [[f"{lr:g}"] + [("diverged" if not np.isfinite(sweep[n][i])
                     else f"{sweep[n][i]:.5f}") for n in sweep]
     for i, lr in enumerate(LRS)],
    ["learning rate"] + list(sweep)))

**What you should see.** Both curves are U-shaped, and Adam's U is *wider*.
That width is the honest summary of what adaptive methods buy you: not a better
optimum, but a much larger range of learning rates that reach a decent one.

Read the two failure directions off the plot, because they look nothing alike
in a training log:

* **Too small** — the loss falls smoothly and stops early. Nothing looks wrong.
  This is the dangerous one, because the run *looks* converged.
* **Too large** — the loss oscillates, then leaves for infinity or `nan`. Ugly,
  obvious, and much easier to diagnose.

## 4 · Stochastic gradient descent: the batch size

Every run above computed the gradient on all two hundred samples at every step,
so the "SGD" in them is plain gradient descent. **Stochastic gradient descent**
computes it on a small random batch instead (L6.1, UDL ch. 6.2). It is the loop
you ran in Ex_05: a `DataLoader` shuffles the data and hands out one batch at a
time, and each batch is one step.

The fair comparison holds the work on the data fixed. One **epoch** is one pass
over all two hundred samples whatever the batch size: one step with a batch of
200, a hundred steps with a batch of 2. Train plain SGD at `lr = 0.05` for 200
epochs with four batch sizes, and record the loss on *all* the data after every
epoch.

### Your turn

In [ ]:
# plain SGD on random batches -----------------------------------------------
import time
from torch.utils.data import DataLoader, TensorDataset

BATCHES = [200, 50, 10, 2]          # 200 is the whole set: plain gradient descent

def run_batches(batch_size, epochs=200, lr=0.05):
    core.set_seed(0)                                        # same initial weights, same shuffles
    model = core.MLP(hidden=(16, 16))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X, Y), batch_size=batch_size, shuffle=True)
    history, steps, t0 = [], 0, time.perf_counter()
    for _ in range(epochs):
        for xb, yb in loader:                               # one random batch, one step
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
            steps += 1
        with torch.no_grad():
            history.append( loss_fn(model(X), Y).item())
    return np.array(history), steps, time.perf_counter() - t0

batch = {}
for bs in BATCHES:
    batch[bs] = run_batches(bs)
    print(f"batch {bs:>3d}: {batch[bs][1]:>6d} steps in {batch[bs][2]:4.1f} s, "
          f"final loss {batch[bs][0][-1]:.5f}")
# ------------------------------------------------------------------------------

In [ ]:
# The same 200 epochs, so the same work on the data, at four batch sizes.
# A smaller batch takes more steps per epoch, and each step is noisier.
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for bs, (hist, steps, secs) in batch.items():
    ax.plot(np.arange(1, len(hist) + 1), hist, lw=1.6, label=f"batch {bs}: {steps} steps")
ax.set_yscale("log")
ax.set_xlabel("epoch (one pass over all 200 samples)"); ax.set_ylabel("loss on all the data")
ax.set_title("Plain SGD, lr = 0.05: the same epochs, four batch sizes")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

print(core.error_table(
    [[str(bs), str(steps), f"{secs:.1f}", f"{hist[-1]:.5f}", f"{hist.min():.5f}",
      f"{hist[-50:].std():.1e}"] for bs, (hist, steps, secs) in batch.items()],
    ["batch size", "steps", "seconds", "final loss", "lowest loss", "jitter, last 50 epochs"]))

**What you should see.** At the same number of epochs the smaller batches end
lower: about 0.047 for the full batch, 0.039 for 50, 0.030 for 10 and 0.007 for
a batch of 2, which took a hundred times as many steps. Each step is cheaper and
rougher, and there are many more of them.

The noise shows in the last two columns. With the full batch the loss falls
every epoch and the final loss is the lowest one. With small batches it jitters
from epoch to epoch, and the final loss sits above the lowest one the run
reached: that is the precision the noise costs near a minimum. With a fixed
learning rate the walk never settles, which is why the learning rate is usually
lowered as training goes on.

The seconds column is the price. A step on two samples is cheaper than a step on
two hundred, but nowhere near a hundred times cheaper, so the batch of 2 takes
far longer for the same epochs. Your seconds will differ from anyone else's; the
ratio is the point. On a GPU a batch of 200 costs little more than a batch of 2,
which is why large networks train on batches of hundreds. The batch size is a
setting of the training, like the learning rate, and not only a memory setting.

## 5 · L-BFGS, and why its interface is different

Everything above takes one gradient and one step. L-BFGS builds an
approximation to the curvature from the last few gradients and uses it to
choose both a direction and a distance — then runs a line search along that
direction, which means **evaluating the loss several times per step**.

So it cannot be handed a single backward pass. It needs a function it can call
repeatedly. That function is the *closure*, and it must do the full
zero-grad / forward / backward each time it is called.

Two consequences worth remembering when you meet this again in L7:

* L-BFGS is a **full-batch** method. It assumes the loss it is shown is the
  same function each time. Mini-batches break that assumption.
* It is fast near a minimum and unreliable far from one, which is exactly the
  opposite of Adam.

In [ ]:
# L-BFGS is a different animal: it uses curvature and needs a
# closure that re-evaluates the loss, because it may call it
# several times per step.
core.set_seed(0)
lbfgs_model = core.MLP(hidden=(16, 16))
opt = torch.optim.LBFGS(lbfgs_model.parameters(), lr=1.0, max_iter=20)

def closure():
    opt.zero_grad()
    loss = loss_fn(lbfgs_model(X), Y)
    loss.backward()
    return loss

lbfgs_history = []
for _ in range(60):
    opt.step(closure)
    with torch.no_grad():
        lbfgs_history.append(loss_fn(lbfgs_model(X), Y).item())

print(f"L-BFGS from a cold start, 60 steps: {lbfgs_history[-1]:.8f}")

## 6 · The handoff Part 2 relies on

Adam is robust far from a solution and slow to polish. L-BFGS is the reverse.
The standard recipe in the PINN literature, and in every exercise in Part 2,
is to use each where it is strong: **Adam to get close, L-BFGS to
finish**.

### Your turn

In [ ]:
# Adam to get close, L-BFGS to finish ---------------------------------------
def adam_then_lbfgs(adam_epochs, lbfgs_steps):
    core.set_seed(0)
    model = core.MLP(hidden=(16, 16))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(adam_epochs):
        opt.zero_grad()
        loss_fn(model(X), Y).backward()
        opt.step()
    opt2 = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=20)
    def closure():
        opt2.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        return loss
    for _ in range(lbfgs_steps):
        opt2.step(closure)
    with torch.no_grad():
        return float(loss_fn(model(X), Y))

final = {}
for name, n_adam, n_lbfgs in [("Adam 2000", 2000, 0),
                              ("L-BFGS 60", 0, 60),
                              ("Adam 1000 -> L-BFGS", 1000, 60),
                              ("Adam 2000 -> L-BFGS", 2000, 60)]:
    final[name] = adam_then_lbfgs(n_adam, n_lbfgs)
    print(f"{name:22s} {final[name]:.3e}")
# ------------------------------------------------------------------------------

In [ ]:
# Adam first, then L-BFGS to finish - against the same budget spent
# entirely on one or the other. This pairing is what the PINN
# notebooks in Part 2 use.
print(core.error_table(
    [[name, f"{loss:.3e}"] for name, loss in final.items()],
    ["recipe", "final training loss"]))

**What you should see.** The chained runs reach a loss several orders of
magnitude below Adam alone, and cold L-BFGS lands somewhere unimpressive — it
is a local method handed a bad starting point.

This is the entire justification for the two-stage training you will type
without thinking for the next six weeks. It is worth having measured it once.

**A caution that matters in Part 2.** A loss of `1e-8` on a PINN residual is
not evidence that the solution is right. It says the network satisfies the
equations you wrote at the points you sampled. If the boundary condition is
wrong, or the collocation points miss a feature, L-BFGS will drive that wrong
problem to machine precision very efficiently. Convergence is not correctness.

## 7 · What this notebook does not show

Say these out loud so the comparison is not oversold:

* **Batches in one section only.** Outside section 4 every run is full-batch,
  so "SGD" there is gradient descent. Batch noise changes the ranking of the
  optimisers, and it can carry a run out of a poor minimum, which a smooth
  noiseless curve like this one does not have.
* **No generalisation.** Training loss only, on noiseless data. The optimiser
  that fits the training set best is not automatically the one you want —
  an overfitted network is the counterexample.
* **One problem, one architecture, one seed for the initial weights.** The
  ordering here is typical, not universal.

## 8 · Save

In [ ]:
# Saved for the report in notebook 06.
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb02_optimisers.npz")
np.savez(path,
         lrs=np.asarray(LRS),
         sweep_sgd=np.asarray(sweep["SGD"], dtype=float),
         sweep_adam=np.asarray(sweep["Adam"], dtype=float),
         hist_sgd=histories["SGD"],
         hist_momentum=histories["SGD + momentum"],
         hist_adam=histories["Adam"],
         hist_lbfgs=np.asarray(lbfgs_history),
         batch_sizes=np.asarray(BATCHES),
         hist_batch=np.stack([batch[bs][0] for bs in BATCHES]),
         final_names=np.array(list(final)),
         final_losses=np.asarray(list(final.values()), dtype=float))
print("wrote", path)
core.saved(path)


## 9 · Before you move on

You should be able to answer these without rerunning anything. Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Plain gradient descent zig-zags in a narrow valley. Using your SGD and
   momentum curves, say what momentum changes — and why the target here is
   noiseless, and what noise would have hidden.
   *→ L6.1 Q9*
2. Which failure mode of a badly chosen learning rate is harder to spot in a
   training log, and why? Answer it for Adam too: what two averages does it
   keep, what does each fix, and why is its loss not monotone?
   *→ L6.1 Q8, Q9*
3. Why does L-BFGS need a closure when Adam does not? Say what L-BFGS uses that
   Adam does not, why it has to be full-batch, and what each call of the closure
   costs in forward and backward passes.
   *→ L6.1 Q7, Q9*
4. Why is cold L-BFGS worse than Adam-then-L-BFGS, given that L-BFGS uses more
   information per step? When is the hand-off worth it?
   *→ L6.1 Q9*


*Write your answers here. You will copy them into the report in notebook 06, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 03](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_03_transfer_and_fine_tuning_light.ipynb)**, where a trained network meets a second machine and most
of what it learned turns out to still be useful.